# ChaoSSM — Phase 15: a fine Lyapunov axis for the frozen reservoir

**Why.** Phase 14 put three points on the Lyapunov axis (ordered -0.30 / edge -0.02 / deep +0.30) and found
ordered ~= edge >> deep ~= linear. That is enough to refute an edge-*specific* benefit, but it leaves the
obvious follow-up unanswered: is forecasting accuracy flat across the whole stable regime, and where exactly
does the benefit die? This notebook answers it by calibrating the SAME frozen cell to seven targets spanning
lambda = -0.60 to +0.30 and measuring accuracy, memory capacity and separation at each.

Everything except the configuration and the sweep is taken verbatim from Phase 14, which is itself the
validated Phase 10 reservoir harness. No cell type, task, protocol or estimator is changed.

**Outputs** go to `results/phase15_lambda_sweep/` and touch no earlier phase:
`phase15_lambda_sweep.csv`, `phase15_summary.csv`, `phase15_lambda_axis.png`.

**Cost.** 7 targets x 5 seeds + 5 linear = 40 training runs at 4000 steps, plus gamma calibration and
diagnostics per run. Expect roughly 25-40 min on a T4.

## Setup (verbatim from Phase 14)

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase15_lambda_sweep"):
    """Resolve the output directory, preferring Google Drive so results survive the runtime.

    Colab's drive.mount() can fail (e.g. ValueError when /content/drive already holds files, or
    when the mount is refused). Earlier versions silently fell back to the VM's local disk, where
    results are lost when the runtime recycles. We now retry the mount once with force_remount and,
    if Drive is still unavailable, fall back to local disk with a loud warning."""
    base=None
    try:
        from google.colab import drive
        try:
            drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}: {e1}); retrying with force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive is NOT mounted ({type(e).__name__}).")
        print(f"[WARNING] Results will be written to {base}")
        print("[WARNING] That is the runtime's local disk -- the files are LOST when the runtime")
        print("[WARNING] recycles. Download them, or copy them to Drive, as soon as the run ends.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
# NOTE: this notebook writes ONLY to its own results/phase15_lambda_sweep/ folder.
# It creates new phase14_*.csv / .png files and does NOT touch or overwrite any existing
# Phase 10/13/14 (or other phase) results. Nothing prior is modified.
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

## Configuration — the only substantive change: a seven-point lambda axis

In [ ]:
SMOKE_TEST=False   # set True for a ~1 min code check before committing the GPU run

# The seven calibration targets. -0.02 and +-0.30 reproduce the Phase 14 points exactly, so this
# sweep is a superset of that experiment and the two can be checked against each other.
LAMBDA_TARGETS=[-0.60,-0.45,-0.30,-0.15,-0.02,0.15,0.30]
def rname(x): return f"lam{x:+.2f}"

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; mlp_mult:int=2
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    lambda_target:dict=field(default_factory=lambda: {rname(x):x for x in LAMBDA_TARGETS})
    lambda_tol:float=0.04          # tighter than Phase 14 (0.05): targets are 0.15 apart here
    calib_steps:int=200; calib_warm:int=60
    inputs:tuple=("random",)
    chaos_regimes:tuple=tuple(rname(x) for x in LAMBDA_TARGETS)
    tasks:tuple=("forecast",)

CFG=Config()
if SMOKE_TEST:
    CFG.steps=30; CFG.seeds=2; CFG.eval_n=40; CFG.calib_steps=30; CFG.calib_warm=10
    CFG.chaos_regimes=(rname(-0.30),rname(-0.02))
    print("*** SMOKE TEST — results are meaningless, this only checks the code runs ***")
print("lambda axis:",list(CFG.chaos_regimes))
print(f"runs = {len(CFG.chaos_regimes)}x{CFG.seeds} chaos + {CFG.seeds} linear = "
      f"{len(CFG.chaos_regimes)*CFG.seeds+CFG.seeds}")

## Cells, calibration, model, tasks, trainer, diagnostics (verbatim from Phase 14)

The `ChaosCell` is unchanged; each point on the axis is reached purely by calibrating gamma to a different target.

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass
    def freeze_input(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"])

In [ ]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    # lo widened from Phase 14's 0.05 to 0.01: the most ordered target (-0.60) needs a smaller gain
    # than any Phase 14 target did. Returns a calibration-success flag so that points the cell
    # cannot actually reach are reported as such rather than silently mislabeled.
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)


# sanity: can the cell actually reach every requested target?
print("calibration reachability check (seed 0):")
for reg in CFG.chaos_regimes:
    c_=ChaosCell(CFG,reg).to(DEVICE); gam,lam,ok=calibrate_gamma(c_,CFG,reg,seed=0)
    flag="ok" if ok else "MISSED"
    print(f"  {reg}  target={CFG.lambda_target[reg]:+.2f} -> gamma={gam:.4f}  measured={lam:+.4f}  [{flag}]")


In [ ]:
class SeqModel(nn.Module):
    def __init__(self,cfg,cond,in_kind,in_dim,out_dim):
        super().__init__(); self.cfg=cfg; self.cond=cond; self.in_kind=in_kind
        if in_kind=="tokens": self.emb=nn.Embedding(in_dim,cfg.d_model)
        else: self.inp=nn.Linear(1,cfg.d_model)
        self.cell=make_cell(cfg,cond)
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        self.achieved_lambda=float("nan"); self.calib_ok=True
        # reservoir arms (chaos, linear) freeze the recurrence; GRU trains fully
        if cond["arm"] in ("chaos","linear"): self.cell.freeze_recurrence()
        # canonical RC: also freeze the input projection AND the encoder
        if cond["input"]=="random" and cond["arm"] in ("chaos","linear"):
            self.cell.freeze_input()
            if in_kind=="tokens":
                self.emb.weight.requires_grad_(False)
            else:
                for p_ in self.inp.parameters(): p_.requires_grad_(False)
    def calibrate(self,cfg):
        if self.cond["arm"]=="chaos":
            _,self.achieved_lambda,self.calib_ok=calibrate_gamma(
                self.cell,cfg,self.cond["regime"],seed=abs(hash(self.cond["regime"]))%1000)
        return self
    def encode(self,x):
        return self.emb(x) if self.in_kind=="tokens" else self.inp(x.unsqueeze(-1))
    def forward(self,x):
        e=self.encode(x); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

In [ ]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
CHANCE=dict(copy=1.0/V, forecast=1.0/CFG.fc_bins)
def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

In [ ]:
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def run_condition(cond,seed,cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    inp,tgt,in_kind,in_dim,out_dim=TASK_FN[cond["task"]](cfg,4,DEVICE,gen)
    model=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE); model.calibrate(cfg)
    evalset=build_eval(cfg,cond["task"])
    params=[p for p in model.parameters() if p.requires_grad]
    n_train=sum(p.numel() for p in params)
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    div=False
    for step in range(cfg.steps):
        inp,tgt,_,_,_=TASK_FN[cond["task"]](cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc(model,evalset) if not div else float("nan")
    return dict(**cond,seed=seed,diverged=div,acc=acc,n_trainable=n_train,
                achieved_lambda=round(float(model.achieved_lambda),3))

In [ ]:
@torch.no_grad()
def memory_capacity(cell, cfg, encode_fn, seed=0, T=1500, washout=200, max_delay=40):
    """Linear short-term memory capacity of a frozen reservoir cell driven by i.i.d. uniform input."""
    dev = next(p.device for p in cell.parameters()); N = cfg.state_dim
    g = torch.Generator(device=dev); g.manual_seed(seed)
    u = torch.rand(T, generator=g, device=dev)          # uniform [0,1]
    drive = encode_fn(u.view(1,-1)).squeeze(0)           # (T, d_model) via model encoder
    h = cell.init_state(1, dev, torch.float32).squeeze(0)
    H=[]
    for t in range(T):
        h = triangle_fold(cell.phi(h) + drive[t] @ cell.B.T) if hasattr(cell,"W_mix") \
            else torch.tanh(h @ cell.W_lin.T + drive[t] @ cell.B.T)
        H.append(h)
    H = torch.stack(H)                                   # (T, N)
    Hd = H[washout:]; um = u[washout:]
    # add bias column
    X = torch.cat([Hd, torch.ones(Hd.shape[0],1,device=dev)],1)
    mc = 0.0; per=[]
    for k in range(1, max_delay+1):
        if k >= Hd.shape[0]-2: break
        Xk = X[k:]; yk = um[:-k]                          # predict u_{t-k} from h_t
        # ridge-regularized least squares, solved robustly (lstsq tolerates rank deficiency)
        lam_ridge = 1e-3
        A = Xk.T @ Xk + lam_ridge*torch.eye(Xk.shape[1],device=dev)
        b = Xk.T @ yk
        w = torch.linalg.lstsq(A, b.unsqueeze(1)).solution.squeeze(1)
        pred = Xk @ w
        # squared correlation
        vp = pred - pred.mean(); vy = yk - yk.mean()
        r2 = (vp@vy)**2 / ((vp@vp)*(vy@vy) + 1e-12)
        r2 = float(torch.clamp(r2,0,1)); per.append(r2); mc += r2
    return mc, per

@torch.no_grad()
def separation(cell, cfg, encode_fn, seed=0, n_streams=200, T=60):
    """Participation-ratio effective rank, (sum s_i)^2 / sum s_i^2, of the final reservoir
    states collected across many independent random input streams. Reported as 'separation'."""
    dev = next(p.device for p in cell.parameters()); N = cfg.state_dim
    g = torch.Generator(device=dev); g.manual_seed(seed)
    finals=[]
    for s in range(n_streams):
        u = torch.rand(T, generator=g, device=dev)
        drive = encode_fn(u.view(1,-1)).squeeze(0)
        h = cell.init_state(1, dev, torch.float32).squeeze(0)
        for t in range(T):
            h = triangle_fold(cell.phi(h) + drive[t] @ cell.B.T) if hasattr(cell,"W_mix") \
                else torch.tanh(h @ cell.W_lin.T + drive[t] @ cell.B.T)
        finals.append(h)
    M = torch.stack(finals)                               # (n_streams, N)
    M = M - M.mean(0, keepdim=True)
    sv = torch.linalg.svdvals(M)
    # effective rank (participation ratio of singular values) — normalized separation proxy
    eff_rank = float((sv.sum()**2) / (sv.pow(2).sum() + 1e-12))
    return eff_rank


## The sweep

For every target on the axis and every seed: train the readout only (canonical protocol) on forecast,
then measure memory capacity and separation on the same frozen, calibrated reservoir. A linear reservoir
is run at the same seeds as the reference.

In [ ]:
def run_lambda_sweep(cfg):
    rows=[]
    for reg in cfg.chaos_regimes:
        for sd in range(cfg.seeds):
            cond=dict(arm="chaos",regime=reg,input="random",task="forecast")
            r=run_condition(cond,sd,cfg)
            torch.manual_seed(sd); np.random.seed(sd)
            g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(sd)
            inp,tgt,in_kind,in_dim,out_dim=TASK_FN["forecast"](cfg,4,DEVICE,g)
            m=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE); m.calibrate(cfg)
            mc,_=memory_capacity(m.cell,cfg,m.encode,seed=sd)
            sep=separation(m.cell,cfg,m.encode,seed=sd)
            rows.append(dict(regime=reg,target=cfg.lambda_target[reg],seed=sd,
                             achieved_lambda=m.achieved_lambda,calib_ok=m.calib_ok,
                             acc=r["acc"],memory_capacity=round(mc,4),separation=round(sep,4),
                             n_trainable=r["n_trainable"],diverged=r["diverged"]))
            print(f"  {reg} s={sd} lam={m.achieved_lambda:+.3f} acc={r['acc']:.4f} "
                  f"MC={mc:.2f} sep={sep:.2f}" + ("" if m.calib_ok else "   [CALIBRATION MISSED]"))
    for sd in range(cfg.seeds):
        cond=dict(arm="linear",regime=cfg.chaos_regimes[0],input="random",task="forecast")
        r=run_condition(cond,sd,cfg)
        torch.manual_seed(sd); np.random.seed(sd)
        m=SeqModel(cfg,cond,"scalar",1,cfg.fc_bins).to(DEVICE)
        mc,_=memory_capacity(m.cell,cfg,m.encode,seed=sd)
        sep=separation(m.cell,cfg,m.encode,seed=sd)
        rows.append(dict(regime="linear",target=float("nan"),seed=sd,
                         achieved_lambda=float("nan"),calib_ok=True,
                         acc=r["acc"],memory_capacity=round(mc,4),separation=round(sep,4),
                         n_trainable=r["n_trainable"],diverged=r["diverged"]))
        print(f"  linear s={sd} acc={r['acc']:.4f} MC={mc:.2f} sep={sep:.2f}")
    return pd.DataFrame(rows)

t0=time.time(); SWEEP=run_lambda_sweep(CFG)
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")
savetable(SWEEP,"phase15_lambda_sweep.csv")

## Summary

In [ ]:
def ci95(x,seed=0):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (float("nan"),float("nan"))
    rng=np.random.default_rng(seed); b=rng.choice(x,size=(10000,len(x)),replace=True).mean(1)
    return float(np.percentile(b,2.5)),float(np.percentile(b,97.5))

lin=SWEEP[SWEEP.regime=="linear"].sort_values("seed")
lin_acc=lin.acc.values; lin_mean=float(np.nanmean(lin_acc))
rows=[]
for reg in CFG.chaos_regimes:
    d=SWEEP[SWEEP.regime==reg].sort_values("seed")
    a=d.acc.values; lo,hi=ci95(a)
    paired=a-lin_acc[:len(a)]
    plo,phi=ci95(paired,seed=1)
    rows.append(dict(regime=reg,target=CFG.lambda_target[reg],
                     achieved_lambda=round(float(np.nanmean(d.achieved_lambda)),3),
                     calib_ok_all=bool(d.calib_ok.all()),
                     mean_acc=round(float(np.nanmean(a)),4), acc_ci=f"[{lo:.3f},{hi:.3f}]",
                     median_acc=round(float(np.nanmedian(a)),4),
                     vs_linear=round(float(np.nanmean(paired)),4),
                     vs_linear_ci=f"[{plo:+.3f},{phi:+.3f}]",
                     wins_over_linear=int((a>lin_acc[:len(a)]).sum()),
                     mean_MC=round(float(np.nanmean(d.memory_capacity)),3),
                     mean_sep=round(float(np.nanmean(d.separation)),3)))
rows.append(dict(regime="linear",target=float("nan"),achieved_lambda=float("nan"),calib_ok_all=True,
                 mean_acc=round(lin_mean,4),acc_ci=f"[{ci95(lin_acc)[0]:.3f},{ci95(lin_acc)[1]:.3f}]",
                 median_acc=round(float(np.nanmedian(lin_acc)),4),vs_linear=0.0,vs_linear_ci="--",
                 wins_over_linear=0,
                 mean_MC=round(float(np.nanmean(lin.memory_capacity)),3),
                 mean_sep=round(float(np.nanmean(lin.separation)),3)))
SUMM=pd.DataFrame(rows)
print(SUMM.to_string(index=False)); savetable(SUMM,"phase15_summary.csv")

ch=SUMM[SUMM.regime!="linear"]
best=ch.loc[ch.mean_acc.idxmax()]
print(f"\nbest point on the axis: {best.regime} (achieved lambda {best.achieved_lambda:+.3f}), "
      f"acc {best.mean_acc:.4f}")
print("reading: a flat plateau across the negative targets that collapses at lambda>0 indicates the")
print("benefit belongs to the stable nonlinear regime; a peak at lambda~0 would support criticality.")
bad=SWEEP[~SWEEP.calib_ok]
if len(bad): print(f"\n[warning] {len(bad)} run(s) missed the calibration tolerance — see calib_ok in the CSV")

## Figure — accuracy, memory capacity and separation along the Lyapunov axis (300 DPI)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.4))
ch=SUMM[SUMM.regime!="linear"].sort_values("target")
x=ch.achieved_lambda.values
for ax,(col,src,title,ylab) in zip(axes,[
        ("mean_acc",None,"Forecast accuracy","accuracy"),
        ("mean_MC",None,"Memory capacity (MC)","MC"),
        ("mean_sep",None,"Separation (eff. rank)","effective rank")]):
    y=ch[col].values
    sd=[float(np.nanstd(SWEEP[SWEEP.regime==r][{"mean_acc":"acc","mean_MC":"memory_capacity",
        "mean_sep":"separation"}[col]].values)) for r in ch.regime]
    ax.errorbar(x,y,yerr=sd,marker="o",ms=7,lw=2,capsize=4,color="#2E7D32")
    ref=float(SUMM[SUMM.regime=="linear"][col].iloc[0])
    ax.axhline(ref,ls="--",color="#777",lw=1.3,label="linear reservoir")
    ax.axvline(0.0,ls=":",color="#B71C1C",lw=1.3,label="chaos onset")
    ax.set_xlabel("achieved autonomous $\\lambda$"); ax.set_ylabel(ylab)
    ax.set_title(title,fontsize=13); ax.grid(alpha=.25); ax.legend(fontsize=9)
fig.suptitle(f"Frozen reservoir along the Lyapunov axis (mean $\\pm$ sd over {CFG.seeds} seeds)",fontsize=13)
fig.tight_layout(); savefig(fig,"phase15_lambda_axis.png"); plt.close(fig)
print("figure saved")

## Rescue the results if Drive was not mounted


In [ ]:
# --- save-guard: if Drive was not mounted, make the results easy to rescue -------------
import shutil, os
if not RESULTS.startswith("/content/drive"):
    _zip = shutil.make_archive("/content/phase15_results","zip",RESULTS)
    print("\n" + "="*78)
    print("Drive was not mounted, so these results live only on this runtime's local disk.")
    print("A zip has been written to:", _zip)
    print("Download it now (Files pane, or run the two lines below) before the runtime recycles:")
    print("  from google.colab import files")
    print(f"  files.download('{_zip}')")
    print("="*78)
else:
    print("\nresults are on Drive:", RESULTS)